In [1]:
import pandas as pd
import json
import datetime
import os

In [2]:
cwd = os.getcwd()
print(f'File Path: {cwd}')

data_path = os.path.join(cwd, 'Transactions')
print(f'Data Location: {data_path}')


File Path: c:\Users\Admin\Documents\06-Dev\Personal ETL Project
Data Location: c:\Users\Admin\Documents\06-Dev\Personal ETL Project\Transactions


In [3]:
folder_files = os.listdir(data_path)
for f in folder_files:
    print(f)

mario_data_20250101.csv
mario_data_20250201.csv
mario_data_20250301.csv
mario_data_20250401.jsonl
mario_data_20250501.jsonl
mario_data_20250601.jsonl


In [4]:
df_peek = pd.read_csv(os.path.join(data_path, 'mario_data_20250101.csv'))

df_peek.head()


,Player Name,Team,World,Vehicle Type,Companion,Kart Racing Rank,Platforming Rank,Boss Battle Rank,Power-Ups Used,Kart Role,Team Points,Lives Lost,Participation in Battle Mode,Mushroom Cup Participation,Power-Ups Owned,Coins Spent in Toad Town,Levels Completed,Times Hit by Enemies,Primary Game
0,Yoshi,Toad Brigade,Yoshi's Island,Comet Bike,pOLTERPUP,A,A,A,12,Drifter,-34.0,0.0,Yes,No,1-Up Mushroom,64,26,4.26,Mario Tennis Aces
1,PeachK,GREEN CAPS,Donut Plains,Circuit Special,kOOPA tROOPA,C,A,B,16,Drifter,149.0,4.0,No,No,"Red Shell, Super Star",335,40,5.00,Mario Tennis Aces
2,Waluigi,NaN,Yoshi's Island,Biddybuggy,Goomba,D,A,C,26,Blocker,174.0,1.0,No,Yes,Green Shell,182,57,5.50,Mario Kart 8 Deluxe
3,Yoshi,Toad Brigade,Star World,Pipe Frame,Goomba,C,D,A,23,Drifter,-1.0,5.0,No,Yes,1-Up Mushroom,333,84,6.00,Super Mario Bros.
4,Bowser Jr.,Koopa Clan,Mushroom Kingdom,Pipe Frame,tOAD,C,C,B,10,Blocker,28.0,2.0,Yes,No,"Red Shell, Banana Peel, Fire Flower",461,55,7.00,Super Mario World


In [5]:
df_list = []

for file in sorted(os.listdir(data_path)):
    file_path = os.path.join(data_path, file)
    print(f"Reading {file}")

    if file.endswith('.csv'):
        temp_df = pd.read_csv(file_path)
    elif file.endswith('.jsonl'):
        with open(file_path, 'r', encoding='utf-8') as f:
            raw_text = f.read().strip()

        if not raw_text:
            print(f"Skipping empty file: {file}")
            continue

        if raw_text.startswith('['):
            records = json.loads(raw_text)
        else:
            records = [json.loads(line) for line in raw_text.splitlines() if line.strip()]

        temp_df = pd.json_normalize(records, sep='_')
    else:
        print(f"Skipping unsupported file: {file}")
        continue

    temp_df['fileName'] = file
    temp_df['loadDatetimeStamp'] = pd.Timestamp.now()

    df_list.append(temp_df)

print("\nCompleted reading all files.")


Reading mario_data_20250101.csv
Reading mario_data_20250201.csv
Reading mario_data_20250301.csv
Reading mario_data_20250401.jsonl
Reading mario_data_20250501.jsonl
Reading mario_data_20250601.jsonl

Completed reading all files.


In [6]:
if df_list:
    df = pd.concat(df_list, ignore_index=True, sort=False)
    print(f'Combined DataFrame shape: {df.shape[0]} rows x {df.shape[1]} columns')
else:
    raise ValueError('No DataFrames created. Check file formats.')


Combined DataFrame shape: 48000 rows x 21 columns


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48000 entries, 0 to 47999
Data columns (total 21 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   Player Name                   38960 non-null  object        
 1   Team                          45040 non-null  object        
 2   World                         48000 non-null  object        
 3   Vehicle Type                  48000 non-null  object        
 4   Companion                     48000 non-null  object        
 5   Kart Racing Rank              38720 non-null  object        
 6   Platforming Rank              48000 non-null  object        
 7   Boss Battle Rank              48000 non-null  object        
 8   Power-Ups Used                48000 non-null  int64         
 9   Kart Role                     48000 non-null  object        
 10  Team Points                   48000 non-null  float64       
 11  Lives Lost                  

In [8]:
output_path = './Medallion Architecture/bronze/bronze_transactions.parquet'
os.makedirs(os.path.dirname(output_path), exist_ok=True)

df.to_parquet(output_path, index=False)
print(f'Bronze Parquet saved to: {output_path}')

Bronze Parquet saved to: ./Medallion Architecture/bronze/bronze_transactions.parquet
